# 04 · Radar counts (Colab)

Each month is a 0.2–2.9 GB ZIP of per-vehicle events. `radares.build_month` downloads one month to the Colab disk, aggregates to 15-minute bins per radar × direction × vehicle class, writes parquet to Drive and deletes the ZIP.

**Zero-cost policy:** use the **CPU** runtime (Runtime → Change runtime type → CPU). GPU is not needed and burns compute units. Never buy extra units.

**Order:** run the smoke test (one month) first. If the PBH server blocks Colab's IPs (403/timeout), run this notebook locally instead.

In [ ]:
# Setup: works on Colab (Drive + private repo) and locally (repo checkout).
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ["BHTRAFFIC_DATA_ROOT"] = "/content/drive/MyDrive/transito_bh"
    token = userdata.get("GH_TOKEN")  # Colab Secrets: fine-grained token, read-only on the repo
    # Clone (not pip-from-git): the package resolves data/ and paper/ relative to the repo root.
    url = f"https://{token}@github.com/Dias0202/Transito_bh.git"
    if os.path.isdir("/content/Transito_bh"):
        !git -C /content/Transito_bh pull -q {url} main
    else:
        !git clone -q {url} /content/Transito_bh
        !git -C /content/Transito_bh remote set-url origin https://github.com/Dias0202/Transito_bh.git
    %pip -q install -e "/content/Transito_bh[geo,viz]"
    sys.path.insert(0, "/content/Transito_bh/src")
    os.chdir("/content/Transito_bh/notebooks")
else:
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd().parent / "src"))

In [ ]:
from pathlib import Path
from bhtraffic.ingest import radares
archives = radares.list_archives()
archives

In [ ]:
# Smoke test: one month end to end before the full loop.
from bhtraffic.config import processed_dir
WORK = Path('/content/radar_zip') if IN_COLAB else Path('radar_zip')
first = archives[0]
if not radares.is_done(first):
    radares.build_month(first, WORK)
import pandas as pd
agg = pd.read_parquet(radares.outputs(radares.month_stem(first))[0])
print(first, agg.shape, agg['bin'].min(), agg['bin'].max())
agg.groupby('classe')['n'].sum()

In [ ]:
# Full loop. Resumable: months with both outputs written are skipped.
for name in archives:
    if radares.is_done(name):
        continue
    print('processing', name, flush=True)
    try:
        radares.build_month(name, WORK)
    except Exception as e:  # one bad month must not stop the rest
        print('FAILED', name, repr(e), flush=True)

In [ ]:
missing = [n for n in archives if not radares.is_done(n)]
print(len(archives) - len(missing), 'of', len(archives), 'months done; missing:', missing)